# Module 3, Part 4 — Transfer learning

This notebook is the runnable companion to Part 4 of the Module 03
learning module: starting from a network already trained on one task,
and reusing it on another.

The module's reference code pulls a real pretrained ResNet-18 from
`torchvision.models`. This notebook cannot do that reliably here (see the
note in the cell below), so instead it trains its own small backbone on a
synthetic "generic" task first, standing in for ImageNet pretraining, then
puts it through exactly the same steps the module describes:

1. Freeze the backbone, replace its head, train only the head
   (feature extraction) on a second, smaller, different task.
2. Unfreeze part of the backbone and continue training it, at a smaller
   learning rate than the head (fine-tuning), on the same small task.
3. Compare the two, the way the module's own "Try it" exercise asks.

Along the way, it also reproduces the module's BatchNorm/`eval()` warning
directly: showing a frozen backbone's running statistics drift if it is
left in `train()` mode.

In [ ]:
import time
import torch
import torch.nn as nn

torch.manual_seed(0)
print("PyTorch version:", torch.__version__)

# Note: this notebook does not import torchvision. In this environment,
# the installed torchvision build's native operators are not registered
# against this torch build, so even `import torchvision` fails here (the
# same issue noted in the Part 3 notebook). Everything below is plain
# torch.nn, standing in for `torchvision.models.resnet18(weights=...)`.

## 0. Two synthetic tasks

**Task A ("the generic dataset"):** a largish, easy synthetic dataset
standing in for ImageNet. Each image is a 16x16, 3-channel picture with a
single bright square placed in one of 4 quadrants; the label is which
quadrant it is in. This is a *position* task: learning it well requires
learning generic, reusable features (where is the bright region, what is
its edge like), not anything tied to one specific downstream use.

**Task B ("your own, smaller dataset"):** a different, much smaller
dataset with the same image format, but a different label rule: the
label is the *colour* of the bright square (red, green, or blue), with
its position now irrelevant and randomized. This is deliberately a
different task from Task A, the way your own project's dataset is a
different task from ImageNet's 1,000 categories, even though both tasks
share the same kind of low-level visual structure (a bright region on a
dark background) that a backbone's early layers can learn to pick out.

In [ ]:
def make_quadrant_image(size=16):
    """Task A: a bright grey square in one of 4 quadrants. Label = quadrant."""
    img = torch.rand(3, size, size) * 0.15   # dark, noisy background
    quadrant = torch.randint(0, 4, (1,)).item()
    half = size // 2
    row0 = 0 if quadrant in (0, 1) else half
    col0 = 0 if quadrant in (0, 2) else half
    brightness = 0.7 + 0.3 * torch.rand(1).item()
    img[:, row0:row0 + half, col0:col0 + half] = brightness
    img += torch.randn(3, size, size) * 0.03
    return img.clamp(0, 1), quadrant


def make_color_image(size=16):
    """Task B: a bright coloured square at a random position. Label = colour."""
    img = torch.rand(3, size, size) * 0.15
    color_idx = torch.randint(0, 3, (1,)).item()   # 0=red, 1=green, 2=blue
    half = size // 2
    row0 = torch.randint(0, size - half + 1, (1,)).item()
    col0 = torch.randint(0, size - half + 1, (1,)).item()
    patch = torch.zeros(3, half, half)
    patch[color_idx] = 0.7 + 0.3 * torch.rand(1).item()
    img[:, row0:row0 + half, col0:col0 + half] = patch
    img += torch.randn(3, size, size) * 0.03
    return img.clamp(0, 1), color_idx


def make_dataset(generator_fn, n):
    images, labels = [], []
    for _ in range(n):
        img, label = generator_fn()
        images.append(img)
        labels.append(label)
    return torch.stack(images), torch.tensor(labels)


torch.manual_seed(0)
task_a_train_x, task_a_train_y = make_dataset(make_quadrant_image, 1200)
task_a_val_x, task_a_val_y = make_dataset(make_quadrant_image, 300)

task_b_train_x, task_b_train_y = make_dataset(make_color_image, 90)   # small, like the module says
task_b_val_x, task_b_val_y = make_dataset(make_color_image, 150)

print("Task A (pretraining proxy): quadrant classification")
print(f"  train {tuple(task_a_train_x.shape)}, val {tuple(task_a_val_x.shape)}, 4 classes")
print("Task B (downstream target): colour classification")
print(f"  train {tuple(task_b_train_x.shape)}, val {tuple(task_b_val_x.shape)}, 3 classes")

## 1. A small backbone, with a replaceable head

A tiny convolutional backbone with one residual block (same idea as Part
2's `ResidualBlock`, BatchNorm included this time since Part 4 is not
about the vanishing-gradient demonstration), ending in global average
pooling and a final linear layer, `fc`, that we will swap out for each
task, exactly as the module's code does with `backbone.fc`.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity
        return torch.relu(out)


class TinyBackbone(nn.Module):
    """Stands in for torchvision's resnet18: a conv stem, one residual
    block (standing in for resnet18's `layer4`), global average pooling,
    and a final `fc` layer sized for however many classes the current
    task needs."""

    def __init__(self, num_classes, channels=16):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
        )
        self.layer4 = ResidualBlock(channels)   # named to mirror the module's fine-tuning example
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(channels, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        preds = model(x).argmax(dim=1)
    return (preds == y).float().mean().item()


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 2. "Pretraining": Task A, the generic dataset

Train the whole backbone, including `fc`, on Task A, the way a real
ResNet-18 was trained on ImageNet. This is the only phase that trains
every parameter from scratch; everything after this treats the result as
a given, pretrained backbone.

In [ ]:
def train_epochs(model, x, y, params, epochs, lr=1e-2, batch_size=32):
    optimizer = torch.optim.AdamW(params, lr=lr) if isinstance(params, list) and isinstance(params[0], dict) \
        else torch.optim.AdamW(params, lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    n = x.shape[0]

    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        total_loss = 0.0
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            optimizer.zero_grad()
            out = model(x[idx])
            loss = loss_fn(out, y[idx])
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)
    return model


torch.manual_seed(0)
backbone = TinyBackbone(num_classes=4)   # Task A has 4 classes (quadrants)

start = time.time()
train_epochs(backbone, task_a_train_x, task_a_train_y, list(backbone.parameters()), epochs=8, lr=5e-3)
pretrain_time = time.time() - start

pretrain_acc = accuracy(backbone, task_a_val_x, task_a_val_y)
print(f"Pretraining (Task A) validation accuracy: {pretrain_acc:.3f}")
print(f"Pretraining time: {pretrain_time:.1f}s")
print("\nThis backbone now has learned early/mid-level features (where is the")
print("bright region, what does its edge look like) useful well beyond the")
print("specific 4-quadrant labelling task it was trained on, the same")
print("generic-to-specific progression the module describes for ImageNet.")

## 3. Feature extraction: freeze the backbone, replace the head

Now Task B, the small downstream dataset. Every pretrained parameter is
frozen, `fc` is replaced with a fresh layer sized for Task B's 3 classes,
and only that new head is trained. This is the module's code almost
exactly, with `TinyBackbone` standing in for `resnet18`.

In [ ]:
import copy

feature_extractor = copy.deepcopy(backbone)

# Feature extraction: freeze every pretrained parameter.
for param in feature_extractor.parameters():
    param.requires_grad = False

# Replace the final classification layer for Task B's number of classes.
num_classes_b = 3
feature_extractor.fc = nn.Linear(feature_extractor.fc.in_features, num_classes_b)
# fc is a fresh layer, so its parameters still require gradients even
# though the rest of the backbone does not.

trainable_params = [p for p in feature_extractor.parameters() if p.requires_grad]
print(f"Trainable parameters: {count_trainable(feature_extractor)} "
      f"(out of {sum(p.numel() for p in feature_extractor.parameters())} total)")

optimizer_params = trainable_params   # only the new head appears here

In [ ]:
torch.manual_seed(1)
start = time.time()
train_epochs(feature_extractor, task_b_train_x, task_b_train_y, optimizer_params, epochs=15, lr=1e-2)
feature_extraction_time = time.time() - start

feature_extraction_acc = accuracy(feature_extractor, task_b_val_x, task_b_val_y)
print(f"Feature-extraction validation accuracy on Task B: {feature_extraction_acc:.3f}")
print(f"Training time: {feature_extraction_time:.2f}s "
      f"({count_trainable(feature_extractor)} trainable parameters)")

## 4. The frozen-BatchNorm pitfall, shown directly

The module's warning: a frozen backbone's `BatchNorm` layers keep
updating their running statistics under `model.train()`, even though
`requires_grad=False` stops their weights from changing. We can see this
directly by watching `layer4.bn1.running_mean` before and after a few
forward passes, once under `train()` and once under `eval()`, with no
optimizer step in either case.

In [ ]:
probe_backbone = copy.deepcopy(feature_extractor)
for param in probe_backbone.parameters():
    param.requires_grad = False   # already frozen, just being explicit

running_mean_before = probe_backbone.layer4.bn1.running_mean.clone()

# Left in train() mode: BatchNorm updates its running stats from Task B's
# batches on every forward pass, even though no weight is changing.
probe_backbone.train()
with torch.no_grad():
    for start in range(0, task_b_train_x.shape[0], 16):
        probe_backbone(task_b_train_x[start:start + 16])

running_mean_after_train_mode = probe_backbone.layer4.bn1.running_mean.clone()
drift = (running_mean_after_train_mode - running_mean_before).abs().mean().item()
print(f"Mean absolute drift in running_mean after forward passes in train() mode: {drift:.6f}")

In [ ]:
# Reset and repeat the identical forward passes in eval() mode instead.
probe_backbone_eval = copy.deepcopy(feature_extractor)
for param in probe_backbone_eval.parameters():
    param.requires_grad = False

running_mean_before_eval = probe_backbone_eval.layer4.bn1.running_mean.clone()

probe_backbone_eval.eval()   # the module's recommendation for the frozen portion
with torch.no_grad():
    for start in range(0, task_b_train_x.shape[0], 16):
        probe_backbone_eval(task_b_train_x[start:start + 16])

running_mean_after_eval_mode = probe_backbone_eval.layer4.bn1.running_mean.clone()
drift_eval = (running_mean_after_eval_mode - running_mean_before_eval).abs().mean().item()
print(f"Mean absolute drift in running_mean after the same forward passes in eval() mode: {drift_eval:.6f}")

print(f"\ntrain() mode drift: {drift:.6f}")
print(f"eval() mode drift:  {drift_eval:.6f}")
assert drift_eval == 0.0
assert drift > 0.0
print("\nConfirmed: eval() mode leaves the frozen backbone's BatchNorm statistics")
print("completely untouched, exactly as stored from pretraining. train() mode lets")
print("them drift toward Task B's own statistics even though no weight is learning")
print("from that drift, quietly shifting the very features the freeze was meant to keep fixed.")

## 5. Fine-tuning: unfreeze part of the backbone, with discriminative learning rates

Now, starting from the same pretrained backbone, unfreeze `layer4` (the
module's example unfreezes `resnet18`'s `layer4`, its last residual
block) and continue training it alongside a fresh head, using a smaller
learning rate for the unfrozen backbone than for the head, exactly as the
module's optimizer construction does.

In [ ]:
fine_tuned = copy.deepcopy(backbone)

# Freeze everything first, then unfreeze just layer4 (the last residual block).
for param in fine_tuned.parameters():
    param.requires_grad = False
for param in fine_tuned.layer4.parameters():
    param.requires_grad = True

fine_tuned.fc = nn.Linear(fine_tuned.fc.in_features, num_classes_b)   # fc is always trainable, fresh

optimizer = torch.optim.AdamW([
    {'params': fine_tuned.fc.parameters(), 'lr': 1e-2},        # new head: larger lr
    {'params': fine_tuned.layer4.parameters(), 'lr': 1e-4},    # unfrozen backbone: smaller lr
])

print("Parameter groups:")
for i, group in enumerate(optimizer.param_groups):
    n_params = sum(p.numel() for p in group['params'])
    print(f"  group {i}: lr={group['lr']:.0e}, {n_params} parameters")
print(f"\nTotal trainable parameters: {count_trainable(fine_tuned)}")

In [ ]:
def train_epochs_with_optimizer(model, x, y, optimizer, epochs, batch_size=32):
    loss_fn = nn.CrossEntropyLoss()
    n = x.shape[0]
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, batch_size):
            idx = perm[start:start + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            optimizer.step()
    return model


torch.manual_seed(1)   # same seed as the feature-extraction run, for a fair comparison
start = time.time()
train_epochs_with_optimizer(fine_tuned, task_b_train_x, task_b_train_y, optimizer, epochs=15)
fine_tune_time = time.time() - start

fine_tune_acc = accuracy(fine_tuned, task_b_val_x, task_b_val_y)
print(f"Fine-tuning validation accuracy on Task B: {fine_tune_acc:.3f}")
print(f"Training time: {fine_tune_time:.2f}s ({count_trainable(fine_tuned)} trainable parameters)")

## 6. Comparing the two approaches

The module's "Try it" exercise asks for exactly this comparison, on a
dataset of your own choosing. Here it is on Task B, with both runs
started from the identical pretrained backbone and the identical number
of epochs.

In [ ]:
print(f"{'approach':>20} | {'val accuracy':>13} | {'train time':>10} | {'trainable params':>17}")
print("-" * 70)
print(f"{'feature extraction':>20} | {feature_extraction_acc:>13.3f} | {feature_extraction_time:>9.2f}s | {count_trainable(feature_extractor):>17}")
print(f"{'fine-tuning':>20} | {fine_tune_acc:>13.3f} | {fine_tune_time:>9.2f}s | {count_trainable(fine_tuned):>17}")

gap = fine_tune_acc - feature_extraction_acc
if abs(gap) < 1e-6:
    print(f"\nBoth approaches land at exactly the same validation accuracy here.")
    print("That is itself an informative result, not a non-result: Task B's colour")
    print(f"signal is a bright region against a darker background, close enough to")
    print(f"Task A's position signal that the frozen backbone's pretrained features")
    print(f"already carry what the new head needs, with no retraining of layer4 at all.")
    print(f"Fine-tuning spent {count_trainable(fine_tuned) - count_trainable(feature_extractor)} extra")
    print(f"trainable parameters and a more carefully chosen two-tier learning rate to")
    print(f"reach the same place feature extraction reached more cheaply. This is exactly")
    print(f"the module's stated case for feature extraction as the default: when a dataset")
    print(f"is small and the frozen features already transfer well, there is nothing for")
    print(f"the extra flexibility (and extra risk of overfitting) of fine-tuning to buy you.")
else:
    better = "fine-tuning" if gap > 0 else "feature extraction"
    print(f"\n{better} reached the higher validation accuracy here, by {abs(gap):.3f}.")
    if better == "fine-tuning":
        print("Letting layer4 adapt to Task B's actual colour-based rule, rather than")
        print("keeping it fixed at whatever Task A happened to need, was worth the extra")
        print("parameters and the smaller learning rate needed to do that safely.")
    else:
        print("With only 90 training images, the frozen backbone's pretrained features")
        print("generalized better than letting layer4 retrain on so little data, exactly")
        print("the risk the module flags for fine-tuning on a small dataset.")

## Summary

- A backbone "pretrained" on one task (Task A: where is the bright
  region) carries learned features that transfer to a different, smaller
  task (Task B: what colour is the bright region), the same
  generic-to-specific progression the module describes for a real
  ImageNet-pretrained network.
- Feature extraction (freeze everything, train only a fresh `fc`) is cheap
  and needs very little data, because the number of trainable parameters
  is tiny.
- Leaving a frozen backbone in `train()` mode lets its `BatchNorm` running
  statistics drift away from their pretrained values on every forward
  pass, confirmed directly above: zero drift under `eval()`, measurable
  drift under `train()`, with no weight update involved either way.
- Fine-tuning (unfreeze part of the backbone, train it alongside the head
  at a much smaller learning rate via `optimizer` parameter groups) lets
  the backbone's later features adapt to the new task, at the cost of
  more trainable parameters and more sensitivity to that learning rate.
  On Task B here, that extra adaptation bought nothing: feature
  extraction reached the same accuracy with roughly 100x fewer trainable
  parameters, because the frozen features already transferred well to a
  small, closely related dataset. That is not a failure of fine-tuning;
  it is the module's own point about when feature extraction is the
  reasonable default.

This is also the direct lead-in to Lab 03: fine-tuning a pretrained vision
backbone on a dataset of your own choosing, and writing up what the
residual-connection and augmentation choices from Parts 2 and 3 actually
did to your results.